# 2. Exploratory Data Analysis

Observed class differences are descriptive and do not establish causation.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.data_validation import load_transactions
df=load_transactions(ROOT/"data"/"creditcard.csv")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [ ]:
counts=df.Class.value_counts().sort_index()
sns.barplot(x=["Legitimate","Fraud"],y=counts.values)
plt.title("Transaction count by actual class"); plt.ylabel("Transactions"); plt.show()
print(f"Fraud prevalence: {df.Class.mean():.6%}")

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.boxplot(data=df,x="Class",y="Amount",showfliers=False,ax=ax[0]); ax[0].set_title("Amount by class (outliers hidden)")
sns.histplot(data=df,x="Amount",hue="Class",bins=60,stat="density",common_norm=False,element="step",ax=ax[1]); ax[1].set_title("Amount distribution by class")
plt.tight_layout(); plt.show(); display(df.groupby("Class").Amount.describe().round(2))

Compare distribution shape and robust statistics; skew and outliers affect means.

In [ ]:
df["elapsed_hour"]=(df.Time//3600).astype(int)
t=df.groupby(["elapsed_hour","Class"]).size().rename("transactions").reset_index()
sns.lineplot(data=t,x="elapsed_hour",y="transactions",hue="Class",marker="o")
plt.title("Transactions by elapsed-hour bucket"); plt.xlabel("Elapsed hours from dataset start"); plt.show()

Elapsed hours are not time of day; no calendar timestamp is provided.

In [ ]:
means=df.groupby("Class")[[f"V{i}" for i in range(1,29)]].mean().T
means.plot(kind="bar",figsize=(14,5)); plt.title("Mean anonymized feature value by actual class"); plt.ylabel("Mean"); plt.tight_layout(); plt.show()

V1–V28 are anonymized PCA features and not directly interpretable. Limitations include historical data, low fraud count, no calendar/customer/merchant context, and uncertain generalization.